## 1. Imports & Setup
Imports required libraries: PyTorch for modelling, NumPy and Pandas for data handling, Matplotlib for visualization, and scikit-learn for scaling.

In [1]:
import math
import copy
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import MinMaxScaler
import random

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# GLOBAL CONFIG — edit here, run once, then execute the rest of the notebook
# ══════════════════════════════════════════════════════════════════════════════

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 55

# ── Target ────────────────────────────────────────────────────────────────────
# Options: 'price day ahead', 'price actual', 'total load actual',
#          'gen_solar', 'gen_wind'
TARGET = 'price day ahead'

# ── Feature engineering toggles ───────────────────────────────────────────────
ADD_FORECAST   = True   # shift(−24) day-ahead load/solar/wind forecasts
ADD_LAGS       = True   # target lag features
LAG_HOURS      = [1, 2, 3, 24, 48]  # which lags to include when ADD_LAGS=True

# ── Sequence config ───────────────────────────────────────────────────────────
WINDOW_SIZE = 48   # hours of look-back context
HORIZON     = 24   # hours ahead to forecast

# ── Training ──────────────────────────────────────────────────────────────────
num_epochs    = 50
# learning_rate = 1e-4
learning_rate = 1e-5

WEIGHT_DECAY  = 1e-4
GRAD_CLIP     = 1.0     # max_norm for gradient clipping

# ── LR scheduler ─────────────────────────────────────────────────────────────
LR_PATIENCE = 10   # epochs without improvement before halving LR
LR_FACTOR   = 0.5

# ── Loss function ─────────────────────────────────────────────────────────────
# Options: 'mae', 'mse', 'custom'
LOSS_FN     = 'mse'
CUSTOM_ALPHA = 0.1   # weight on JSD term (only used when LOSS_FN='custom')
CUSTOM_BETA  = 0.01  # weight on smoothness term

# ── DataLoader ────────────────────────────────────────────────────────────────
BATCH_SIZE  = 256 if torch.cuda.is_available() else 64
NUM_WORKERS = 4   if torch.cuda.is_available() else 0
PIN_MEMORY  = torch.cuda.is_available()


In [ ]:
# ── Model selection ───────────────────────────────────────────────────────────
# Options: 'transformer', 'cnn_bilstm', 'lstm', 'bilstm', 'gru', 'bigru'
MODEL_TYPE = 'gru'

# ── Transformer architecture ──────────────────────────────────────────────────
D_MODEL         = 64
N_HEAD          = 4
NUM_ENC_LAYERS  = 2
DIM_FEEDFORWARD = 256
DROPOUT         = 0.1

# ── CNN-BiLSTM architecture ───────────────────────────────────────────────────
CNN_FILTERS     = 64
CNN_KERNEL_SIZE = 3

# ── LSTM / BiLSTM / GRU / BiGRU (shared params) ──────────────────────────────
RNN_HIDDEN      = 128   # hidden size
# RNN_HIDDEN      = 64   # hidden size

# RNN_LAYERS      = 2     # number of stacked layers
RNN_LAYERS      = 1     # number of stacked layers

RNN_DROPOUT     = 0.1   # dropout between layers

### Manual seed 

In [ ]:
# SEED = 55

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if torch.backends.mps.is_available():
    torch.mps.manual_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

print(f"Random seed set to {SEED}")

## 2. Data Loading
Loads the merged energy and weather dataset (35,064 rows x 62 columns) - hourly observations with energy production/consumption and weather features.

In [ ]:
energy_weather = pd.read_csv('../../data/processed/energy_weather_merged.csv')
energy_weather.info()
print(f"\nShape: {energy_weather.shape}")

## 3. Feature & Target Definition
Set `TARGET` to whichever column you want to predict. Then toggle features on/off by commenting or uncommenting lines in `feature_cols` — the target is automatically excluded. Columns are grouped by category to make it easy to swap between simple and full setups.

In [ ]:
# ── Features ─────────────────────────────────────────────────────────────────
# Comment out any column you don't want as an input feature.
# TARGET is automatically removed from this list at the bottom.

feature_cols = [

    # ── Forecasts (day-ahead signals published before the target hour) ────────
    # 'forecast solar day ahead',
    # 'forecast wind onshore day ahead',
    # 'total load forecast',

    # ── Grid & generation ─────────────────────────────────────────────────────
    'total load actual',
    'gen_solar',
    'gen_wind',
    'gen_hydro',
    'gen_pumped_hydro',
    'gen_fossil',
    'gen_nuclear',
    'gen_other',

    # ── Other price series ────────────────────────────────────────────────────
    # 'price actual',

    # ── Weather — temperature (per city) ─────────────────────────────────────
    # 'temp_Barcelona',
    # 'temp_Bilbao',
    # 'temp_Madrid',
    # 'temp_Seville',
    # 'temp_Valencia',
    # 'temp_avg',                   # average across cities — redundant with above

    # ── Weather — wind speed (per city) ──────────────────────────────────────
    # 'wind_speed_Barcelona',
    # 'wind_speed_Bilbao',
    # 'wind_speed_Madrid',
    # 'wind_speed_Seville',
    # 'wind_speed_Valencia',

    # ── Weather — cloud cover (per city) ─────────────────────────────────────
    # 'clouds_all_Barcelona',
    # 'clouds_all_Bilbao',
    # 'clouds_all_Madrid',
    # 'clouds_all_Seville',
    # 'clouds_all_Valencia',

    # ── Weather — rain 3h (per city) ─────────────────────────────────────────
    # 'rain_3h_Barcelona',
    # 'rain_3h_Bilbao',
    # 'rain_3h_Madrid',
    # 'rain_3h_Seville',
    # 'rain_3h_Valencia',

    # ── Weather — rain 1h (per city) — often sparse, consider removing ────────
    # 'rain_1h_Barcelona',
    # 'rain_1h_Bilbao',
    # 'rain_1h_Madrid',
    # 'rain_1h_Seville',
    # 'rain_1h_Valencia',

    # ── Weather — snow 3h (per city) — mostly zero outside winter ────────────
    # 'snow_3h_Barcelona',
    # 'snow_3h_Bilbao',
    # 'snow_3h_Madrid',
    # 'snow_3h_Seville',
    # 'snow_3h_Valencia',

    # ── Weather — wind direction (per city) — circular, consider encoding ─────
    # 'wind_deg_Barcelona',
    # 'wind_deg_Bilbao',
    # 'wind_deg_Madrid',
    # 'wind_deg_Seville',
    # 'wind_deg_Valencia',

    # ── Weather — humidity (per city) ────────────────────────────────────────
    # 'humidity_Barcelona',
    # 'humidity_Bilbao',
    # 'humidity_Madrid',
    # 'humidity_Seville',
    # 'humidity_Valencia',

    # ── Weather — pressure (per city) ────────────────────────────────────────
    # 'pressure_Barcelona',
    # 'pressure_Bilbao',
    # 'pressure_Madrid',
    # 'pressure_Seville',
    # 'pressure_Valencia',
]

# Remove TARGET from features if it appears in the list above
feature_cols = [c for c in feature_cols if c != TARGET]

print(f"Target   : {TARGET}")
print(f"Features ({len(feature_cols)}):")
for col in feature_cols:
    print(f"  {col}")

## 4. Feature Engineering
Adds cyclical encodings of hour-of-day, day-of-week, and month (sine/cosine pairs) so the model can learn daily and weekly price patterns, plus a binary weekend flag. No lag features are created.

In [ ]:
energy_weather['time'] = pd.to_datetime(energy_weather['time'])
energy_weather['year'] = energy_weather['time'].dt.year

# UTC timestamps include one row from 2014-12-31 23:00 UTC
# (= first hour of 2015 in Central European Time). Drop it so
# the year column contains only complete calendar years 2015-2018.
energy_weather = energy_weather[energy_weather['year'] >= 2015].reset_index(drop=True)

hour  = energy_weather['time'].dt.hour
dow   = energy_weather['time'].dt.dayofweek
month = energy_weather['time'].dt.month

energy_weather['hour_sin']   = np.sin(2 * np.pi * hour        / 24)
energy_weather['hour_cos']   = np.cos(2 * np.pi * hour        / 24)
energy_weather['day_sin']    = np.sin(2 * np.pi * dow         / 7)
energy_weather['day_cos']    = np.cos(2 * np.pi * dow         / 7)
energy_weather['month_sin']  = np.sin(2 * np.pi * (month - 1) / 12)
energy_weather['month_cos']  = np.cos(2 * np.pi * (month - 1) / 12)
energy_weather['is_weekend'] = (dow >= 5).astype(float)

time_features = [
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos',
    'month_sin', 'month_cos', 'is_weekend',
]
feature_cols = feature_cols + time_features


In [ ]:
# ── Forward-looking forecast features (optional) ──────────────────────────────

if ADD_FORECAST:
    energy_weather['total_load_forecast_ahead'] = \
        energy_weather['total load forecast'].shift(-24)
    
    energy_weather['forecast_solar_ahead'] = \
        energy_weather['forecast solar day ahead'].shift(-24)
    
    energy_weather['forecast_wind_ahead'] = \
        energy_weather['forecast wind onshore day ahead'].shift(-24)
    
    forecast_cols = [
        'total_load_forecast_ahead',
        'forecast_solar_ahead',
        'forecast_wind_ahead'
    ]
    feature_cols = feature_cols + forecast_cols
    
    print(f"Forward forecast features added: {forecast_cols}")
    print(f"Total features now: {len(feature_cols)}")
else:
    print("Forward forecast features: disabled")



# ── 3. Lag features ───────────────────────────────────────────────────────────

if ADD_LAGS:
    # lag_hours = [1, 2, 3, 24, 48] (defined in global config)
    lag_cols  = []
    for lag in LAG_HOURS:
        col_name = f"{TARGET.replace(' ', '_')}_lag_{lag}"
        energy_weather[col_name] = energy_weather[TARGET].shift(lag)
        lag_cols.append(col_name)
    feature_cols = feature_cols + lag_cols
    print(f"Lag features added  : {len(lag_cols)}")
    print(f"Total features now  : {len(feature_cols)}")
else:
    lag_cols = []
    print("Lag features: disabled")


In [ ]:
# ── 4. Single dropna — covers everything ─────────────────────────────────────
energy_weather = energy_weather.dropna(
    subset=feature_cols + [TARGET]
).reset_index(drop=True)

print(f"Shape after feature engineering : {energy_weather.shape}")
print(f"Total input features            : {len(feature_cols)}")
print(f"Years in data                   : {sorted(energy_weather['year'].unique())}")

In [ ]:
feature_cols

## 5. Train / Validation / Test Split
Splits by calendar year in strict chronological order: **first 2 years -> train, year 3 -> validation, year 4 -> test**. `MinMaxScaler` is fit only on the training set to prevent data leakage. The target has its own scaler so predictions can be inverse-transformed back to EUR/MWh.

In [ ]:
years = sorted(energy_weather['year'].unique())

train_df = energy_weather[energy_weather['year'].isin(years[:2])].reset_index(drop=True)
val_df   = energy_weather[energy_weather['year'] == years[2]].reset_index(drop=True)
test_df  = energy_weather[energy_weather['year'] == years[3]].reset_index(drop=True)

# ── Split feature_cols into two groups ───────────────────────────────────────
no_scale_features = time_features                                        # already in [-1, 1]
scale_features    = [c for c in feature_cols if c not in no_scale_features]  # everything else

# Features — fit scaler on train only, applied to scale_features only
feat_scaler = MinMaxScaler()

def scale_features_df(df):
    scaled_part    = feat_scaler.transform(df[scale_features].values)
    noscale_part   = df[no_scale_features].values
    return np.concatenate([scaled_part, noscale_part], axis=1)

feat_scaler.fit(train_df[scale_features].values)

X_train = scale_features_df(train_df)
X_val   = scale_features_df(val_df)
X_test  = scale_features_df(test_df)

# Target — fit on train only
tgt_scaler = MinMaxScaler()
y_train = tgt_scaler.fit_transform(train_df[[TARGET]].values).ravel()
y_val   = tgt_scaler.transform(val_df[[TARGET]].values).ravel()
y_test  = tgt_scaler.transform(test_df[[TARGET]].values).ravel()

print(f"Train : {years[:2]}  ->  {len(train_df):,} rows")
print(f"Val   : {years[2]}   ->  {len(val_df):,} rows")
print(f"Test  : {years[3]}   ->  {len(test_df):,} rows")
print(f"Features scaled    : {len(scale_features)}")
print(f"Features not scaled: {len(no_scale_features)} (sin/cos + is_weekend)")


### Check the distribution of the target variable for skewness/outliers 

In [ ]:
import matplotlib.pyplot as plt
train_df[TARGET].hist(bins=100)
plt.title('Price distribution')
plt.show()

print(train_df[TARGET].describe())
print(f"Skew: {train_df[TARGET].skew():.2f}")

## 6. Split Verification
Confirms array shapes and date ranges for each split.

In [ ]:
print(f"X_train : {X_train.shape}    y_train : {y_train.shape}")
print(f"X_val   : {X_val.shape}      y_val   : {y_val.shape}")
print(f"X_test  : {X_test.shape}     y_test  : {y_test.shape}")

print(f"\nTrain : {train_df['time'].iloc[0]}  ->  {train_df['time'].iloc[-1]}")
print(f"Val   : {val_df['time'].iloc[0]}  ->  {val_df['time'].iloc[-1]}")
print(f"Test  : {test_df['time'].iloc[0]}  ->  {test_df['time'].iloc[-1]}")

print(f"\nFeature scale check -- X_train : min={X_train.min():.3f}, max={X_train.max():.3f}")
print(f"Target  scale check -- y_train : min={y_train.min():.3f}, max={y_train.max():.3f}")

## 7. Configuration
Hyperparameters for the data pipeline and Transformer model.

| Parameter | Value | Notes |
|---|---|---|
| `WINDOW_SIZE` | 48 | Hours of look-back the Transformer attends over |
| `HORIZON` | 24 | Hours ahead to forecast |
| `D_MODEL` | 64 | Internal embedding dimension (must be divisible by `N_HEAD`) |
| `N_HEAD` | 4 | Self-attention heads |
| `NUM_ENC_LAYERS` | 2 | Stacked encoder blocks |
| `DIM_FEEDFORWARD` | 256 | FFN hidden dimension inside each encoder block |
| `DROPOUT` | 0.1 | Applied after attention and FFN sub-layers |

In [ ]:
INPUT_SIZE      = len(feature_cols)

print(f"Input window     : {WINDOW_SIZE} hours")
print(f"Forecast horizon : {HORIZON} hours")
print(f"Input features   : {INPUT_SIZE}")
print(f"d_model={D_MODEL}, heads={N_HEAD}, enc_layers={NUM_ENC_LAYERS}, ffn={DIM_FEEDFORWARD}")

## 8. Dataset & DataLoader Construction
`EnergyDataset` slices the data into overlapping windows. For each index the input is the past `WINDOW_SIZE` hours of features and the target is the next `HORIZON` hours of day-ahead price.

In [ ]:
class EnergyDataset(Dataset):
    def __init__(self, X, y, window_size=WINDOW_SIZE, horizon=HORIZON):
        self.X           = torch.tensor(X, dtype=torch.float32)
        self.y           = torch.tensor(y, dtype=torch.float32)
        self.window_size = window_size
        self.horizon     = horizon

    def __len__(self):
        return len(self.X) - self.window_size - self.horizon + 1

    def __getitem__(self, idx):
        x_seq = self.X[idx : idx + self.window_size]
        y_seq = self.y[idx + self.window_size : idx + self.window_size + self.horizon]
        return x_seq, y_seq

In [ ]:
train_dataset = EnergyDataset(X_train, y_train)
val_dataset   = EnergyDataset(X_val,   y_val)
test_dataset  = EnergyDataset(X_test,  y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

x_sample, y_sample = next(iter(train_loader))
print(f"X batch : {tuple(x_sample.shape)}  ->  expected (batch, {WINDOW_SIZE}, {INPUT_SIZE})")
print(f"y batch : {tuple(y_sample.shape)}  ->  expected (batch, {HORIZON})")

## 9. Model Architecture

A standard Transformer **encoder-only** pipeline:

1. **Input projection** - linear layer maps each timestep from `INPUT_SIZE` raw features to `D_MODEL` dimensions.
2. **Positional encoding** - adds sinusoidal position signals so the model knows the order of timesteps (Transformers have no built-in sense of sequence order).
3. **Transformer encoder** - `NUM_ENC_LAYERS` stacked encoder blocks, each containing:
   - Multi-head self-attention (`N_HEAD` heads, each of size `D_MODEL // N_HEAD`)
   - Feed-forward network (`DIM_FEEDFORWARD` hidden units)
   - Layer norm + residual connections (built into PyTorch's `TransformerEncoderLayer`)
4. **Output head** - takes the **last timestep's** representation and projects it to `HORIZON` output values via a single linear layer.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe       = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class EnergyTransformer(nn.Module):
    def __init__(self, input_size, d_model=D_MODEL, nhead=N_HEAD,
                 num_encoder_layers=NUM_ENC_LAYERS,
                 dim_feedforward=DIM_FEEDFORWARD,
                 dropout=DROPOUT, horizon=HORIZON):
        super().__init__()
        self.input_proj   = nn.Linear(input_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, dropout)
        encoder_layer     = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True,
        )
        self.transformer  = nn.TransformerEncoder(
            encoder_layer, num_layers=num_encoder_layers)
        self.output_head  = nn.Linear(d_model, horizon)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pos_encoding(x)
        x = self.transformer(x)
        x = x[:, -1, :]
        return self.output_head(x)


class CNNBiLSTM(nn.Module):
    def __init__(self, input_size,
                 cnn_filters=CNN_FILTERS,
                 cnn_kernel_size=CNN_KERNEL_SIZE,
                 lstm_hidden=RNN_HIDDEN,
                 lstm_layers=RNN_LAYERS,
                 lstm_dropout=RNN_DROPOUT,
                 dropout=DROPOUT,
                 horizon=HORIZON):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels=input_size,
                      out_channels=cnn_filters,
                      kernel_size=cnn_kernel_size,
                      padding=cnn_kernel_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.bilstm = nn.LSTM(
            input_size=cnn_filters,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            dropout=lstm_dropout if lstm_layers > 1 else 0,
            batch_first=True,
            bidirectional=True,
        )
        self.dropout     = nn.Dropout(dropout)
        self.output_head = nn.Linear(lstm_hidden * 2, horizon)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.cnn(x)
        x = x.permute(0, 2, 1)
        x, _ = self.bilstm(x)
        x = x[:, -1, :]
        x = self.dropout(x)
        return self.output_head(x)


class EnergyRNN(nn.Module):
    """
    Unified class for LSTM, BiLSTM, GRU, BiGRU.
    Controlled via rnn_type and bidirectional flags.
    """
    def __init__(self, input_size,
                 rnn_type='lstm',          # 'lstm' or 'gru'
                 bidirectional=False,
                 hidden_size=RNN_HIDDEN,
                 num_layers=RNN_LAYERS,
                 rnn_dropout=RNN_DROPOUT,
                 dropout=DROPOUT,
                 horizon=HORIZON):
        super().__init__()

        self.bidirectional = bidirectional
        rnn_class          = nn.LSTM if rnn_type == 'lstm' else nn.GRU

        self.rnn = rnn_class(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=rnn_dropout if num_layers > 1 else 0,
            batch_first=True,
            bidirectional=bidirectional,
        )

        self.dropout = nn.Dropout(dropout)

        # bidirectional doubles the output size
        rnn_output_size  = hidden_size * 2 if bidirectional else hidden_size
        self.output_head = nn.Linear(rnn_output_size, horizon)

    def forward(self, x):
        # x: (batch, window_size, input_size)
        x, _ = self.rnn(x)        # (batch, window_size, hidden*directions)
        x = x[:, -1, :]           # last timestep
        x = self.dropout(x)
        return self.output_head(x)


# ── Model factory ─────────────────────────────────────────────────────────────
def build_model(input_size):
    if MODEL_TYPE == 'transformer':
        return EnergyTransformer(input_size=input_size)

    elif MODEL_TYPE == 'cnn_bilstm':
        return CNNBiLSTM(input_size=input_size)

    elif MODEL_TYPE == 'lstm':
        return EnergyRNN(input_size=input_size,
                         rnn_type='lstm', bidirectional=False)

    elif MODEL_TYPE == 'bilstm':
        return EnergyRNN(input_size=input_size,
                         rnn_type='lstm', bidirectional=True)

    elif MODEL_TYPE == 'gru':
        return EnergyRNN(input_size=input_size,
                         rnn_type='gru', bidirectional=False)

    elif MODEL_TYPE == 'bigru':
        return EnergyRNN(input_size=input_size,
                         rnn_type='gru', bidirectional=True)

    else:
        raise ValueError(
            f"MODEL_TYPE must be one of: "
            f"'transformer', 'cnn_bilstm', 'lstm', 'bilstm', 'gru', 'bigru'. "
            f"Got '{MODEL_TYPE}'"
        )

In [ ]:
# ── Parameter count check ─────────────────────────────────────────────────────
_model       = build_model(input_size=INPUT_SIZE)
total_params = sum(p.numel() for p in _model.parameters()
                   if p.requires_grad)
print(f"{MODEL_TYPE.upper()} -- {total_params:,} trainable parameters")
print(_model)

## 10. Device Selection
Selects the best available compute device: CUDA (NVIDIA GPU) -> MPS (Apple Silicon) -> CPU.

In [ ]:
device = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else
    'cpu'
)
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 11. Training & Evaluation Functions
`train_model()` uses **MSE loss** for training (smoother gradients than MAE), an Adam optimizer, and a `ReduceLROnPlateau` scheduler that halves the learning rate after 5 epochs without improvement. `test_model()` evaluates on the test set and inverse-transforms predictions back to original price units (EUR/MWh), reporting **MAE** and **RMSE** as the final metrics.

### Custom Loss Function 

In [ ]:
import copy

# ── Custom Loss ───────────────────────────────────────────────────────────────
class CustomLoss(nn.Module):
    def __init__(self, alpha=0.1, beta=0.01):
        super(CustomLoss, self).__init__()
        self.alpha = alpha
        self.beta  = beta

    def forward(self, y_pred, y_true):
        y_pred = y_pred.squeeze()
        y_true = y_true.squeeze()

        # ── MAE ──────────────────────────────────────────
        mae = torch.mean(torch.abs(y_true - y_pred))

        # ── JSD ──────────────────────────────────────────
        y_true_soft = torch.softmax(y_true, dim=0)
        y_pred_soft = torch.softmax(y_pred, dim=0)
        m           = 0.5 * (y_true_soft + y_pred_soft)
        kl1         = torch.sum(
            y_true_soft * torch.log(y_true_soft / (m + 1e-8) + 1e-8))
        kl2         = torch.sum(
            y_pred_soft * torch.log(y_pred_soft / (m + 1e-8) + 1e-8))
        jsd         = 0.5 * (kl1 + kl2)

        # ── Smoothness ────────────────────────────────────
        smoothness = torch.mean((y_pred[1:] - y_pred[:-1]) ** 2)

        return mae + self.alpha * jsd + self.beta * smoothness

### Training loop 

In [ ]:
def train_model(model, train_loader, val_loader,
                num_epochs=50, lr=1e-3, device='cpu',
                print_every=1, loss_fn='mae',
                weight_decay=1e-4, alpha=0.1, beta=0.01):

    model     = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=10, factor=0.5
    )

    # ── Loss selection ────────────────────────────────────────────────────────
    if loss_fn == 'mae':
        criterion  = nn.L1Loss()
        loss_label = 'MAE'
    elif loss_fn == 'mse':
        criterion  = nn.MSELoss()
        loss_label = 'MSE'
    elif loss_fn == 'custom':
        criterion  = CustomLoss(alpha=alpha, beta=beta)
        loss_label = f'Custom (α={alpha} β={beta})'
    else:
        raise ValueError(f"loss_fn must be 'mae', 'mse' or 'custom', got '{loss_fn}'")

    print(f"Training with {loss_label} loss")

    best_val_loss = float('inf')
    best_weights  = copy.deepcopy(model.state_dict())
    history       = {'train_loss': [], 'val_loss': [], 'lr': []}

    for epoch in range(num_epochs):
        model.train()
        train_losses = []
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                val_losses.append(criterion(model(X_batch), y_batch).item())

        train_loss = float(np.mean(train_losses))
        val_loss   = float(np.mean(val_losses))
        scheduler.step(val_loss)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['lr'].append(optimizer.param_groups[0]['lr'])

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights  = copy.deepcopy(model.state_dict())

        if (epoch + 1) % print_every == 0:
            print(f"Epoch {epoch+1:3d}/{num_epochs}  "
                  f"train={train_loss:.4f}  val={val_loss:.4f}  "
                  f"lr={optimizer.param_groups[0]['lr']:.2e}")

    model.load_state_dict(best_weights)
    print(f"\nBest val loss ({loss_label}): {best_val_loss:.4f}")
    return model, history


## 12. Training Execution
Trains the Transformer on the 2-year training set, monitors on the validation year, then evaluates on the held-out test year.

In [ ]:
import random
import copy
import time


# ── Random Search Config ──────────────────────────────────────────────────────
N_TRIALS = 10   # number of random configs to try

search_space = {
    'lr'         : [1e-5, 5e-5, 1e-4],
    'hidden'     : [64, 128, 256],
    'layers'     : [1, 2],
    'dropout'    : [0.0, 0.1, 0.2, 0.3],
    'rnn_dropout': [0.0, 0.1, 0.2],
}



# ── Run Random Search ─────────────────────────────────────────────────────────
random.seed(SEED)
search_results = []

for trial in range(N_TRIALS):
    # sample random config
    config = {k: random.choice(v) for k, v in search_space.items()}

    print(f"\nTrial {trial+1}/{N_TRIALS}  |  {config}")

    rnn_type      = 'gru' if 'gru' in MODEL_TYPE else 'lstm'
    bidirectional = 'bi' in MODEL_TYPE

    # build model with this config
    trial_model = EnergyRNN(
        input_size    = INPUT_SIZE,
        rnn_type      = rnn_type,
        bidirectional = bidirectional,
        hidden_size   = config['hidden'],
        num_layers    = config['layers'],
        rnn_dropout   = config['rnn_dropout'] if config['layers'] > 1 else 0,
        dropout       = config['dropout'],
        horizon       = HORIZON,
    ).to(device)

    # train
    try:
        trained_trial, history_trial = train_model(
            trial_model, train_loader, val_loader,
            num_epochs   = num_epochs,
            lr           = config['lr'],
            device       = device,
            loss_fn      = LOSS_FN,
            weight_decay = WEIGHT_DECAY,
            print_every  = 999,   # suppress per-epoch output
        )
        best_val = min(history_trial['val_loss'])
        print(f"  → best val loss: {best_val:.4f}")

    except Exception as e:
        print(f"  → failed: {e}")
        best_val = float('inf')

    search_results.append({
        'trial'   : trial + 1,
        'config'  : config,
        'val_loss': best_val,
    })

# ── Print Results ─────────────────────────────────────────────────────────────
search_results = sorted(search_results, key=lambda x: x['val_loss'])

print(f"\n{'='*65}")
print(f"Random Search Results — GRU  ({N_TRIALS} trials)")
print(f"{'='*65}")
print(f"{'Trial':>6} {'LR':>8} {'Hidden':>8} {'Layers':>8} "
      f"{'Dropout':>9} {'Val Loss':>10}")
print(f"{'─'*65}")
for r in search_results:
    c = r['config']
    print(f"{r['trial']:>6} {c['lr']:>8.0e} {c['hidden']:>8} "
          f"{c['layers']:>8} {c['dropout']:>9.2f} {r['val_loss']:>10.4f}")

print(f"\nBest config:")
best = search_results[0]
for k, v in best['config'].items():
    print(f"  {k:<15} : {v}")
print(f"  {'val_loss':<15} : {best['val_loss']:.4f}")

In [ ]:
# ── Retrain Best Config ───────────────────────────────────────────────────────
best_config = search_results[0]['config']
print(f"Retraining best GRU config: {best_config}")

rnn_type      = 'gru' if 'gru' in MODEL_TYPE else 'lstm'
bidirectional = 'bi' in MODEL_TYPE

model = EnergyRNN(
    input_size    = INPUT_SIZE,
    rnn_type      = rnn_type,
    bidirectional = bidirectional,
    hidden_size   = best_config['hidden'],
    num_layers    = best_config['layers'],
    rnn_dropout   = best_config['rnn_dropout'] if best_config['layers'] > 1 else 0,
    dropout       = best_config['dropout'],
    horizon       = HORIZON,
).to(device)

t0 = time.time()
trained_model, history = train_model(
    model, train_loader, val_loader,
    num_epochs   = num_epochs,
    lr           = best_config['lr'],
    device       = device,
    loss_fn      = LOSS_FN,
    weight_decay = WEIGHT_DECAY,
    print_every  = 1,
)
elapsed = time.time() - t0
mins, secs = divmod(int(elapsed), 60)
print(f"\nTotal training time: {mins}m {secs}s")

In [ ]:
import time

assert X_train.shape[1] == INPUT_SIZE, (
    f"Feature mismatch: X_train has {X_train.shape[1]} features but "
    f"INPUT_SIZE={INPUT_SIZE}. Re-run cells 6 → 8 → 13 → 17 before training."
)

model = build_model(input_size=INPUT_SIZE).to(device)

t0 = time.time()
trained_model, history = train_model(
    model, train_loader, val_loader,
    num_epochs=num_epochs, lr=learning_rate, device=device, loss_fn=LOSS_FN, weight_decay=WEIGHT_DECAY
)

elapsed = time.time() - t0

mins, secs = divmod(int(elapsed), 60)
print(f"\nTotal training time: {mins}m {secs}s")

## 13. Testing & Evaluation 

### Helper functions for testing/evaluation 

In [ ]:
def compute_metrics(preds_orig, actuals_orig):
    mae  = float(np.mean(np.abs(preds_orig - actuals_orig)))
    rmse = float(np.sqrt(np.mean((preds_orig - actuals_orig) ** 2)))
    mape = float(np.mean(np.abs((actuals_orig - preds_orig) /
                                 np.abs(actuals_orig).clip(min=1.0))) * 100)
    mae_per_horizon = np.mean(np.abs(preds_orig - actuals_orig), axis=0)
    return mae, rmse, mape, mae_per_horizon


def print_metrics(mae, rmse, mape, mae_per_horizon):
    print(f"MAE  : {mae:.3f} EUR/MWh  (on average, predictions are off by {mae:.1f} EUR/MWh)")
    print(f"RMSE : {rmse:.3f} EUR/MWh  (higher than MAE means occasional big misses)")
    print(f"MAPE : {mape:.2f}%          (on average, predictions are off by {mape:.1f}% of the actual price)")
    # print(f"\nPer-horizon MAE (EUR/MWh)  — does error grow the further ahead we predict?")
    # for h, m in enumerate(mae_per_horizon, 1):
    #     bar = '█' * int(m)
    #     print(f"  h+{h:02d} : {m:6.3f}  {bar}")

### Non-overlapping version 

In [ ]:
def forecast_production(model, X_test, y_test, tgt_scaler, device='cpu',
                        window_size=WINDOW_SIZE, horizon=HORIZON):
    model.eval()
    X_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_tensor = torch.tensor(y_test, dtype=torch.float32)
    preds, actuals = [], []

    with torch.no_grad():
        for start in range(0, len(X_test) - window_size - horizon + 1, horizon):
            x_window = X_tensor[start : start + window_size].unsqueeze(0).to(device)
            y_window = y_tensor[start + window_size : start + window_size + horizon].numpy()
            pred     = model(x_window).cpu().numpy()
            preds.append(pred)
            actuals.append(y_window)

    preds   = np.concatenate(preds,  axis=0)   # (N_days, 24)
    actuals = np.stack(actuals,      axis=0)   # (N_days, 24)

    preds_orig   = tgt_scaler.inverse_transform(preds.reshape(-1, 1)).reshape(preds.shape)
    actuals_orig = tgt_scaler.inverse_transform(actuals.reshape(-1, 1)).reshape(actuals.shape)

    mae, rmse, mape, mae_per_horizon = compute_metrics(preds_orig, actuals_orig)

    print(f"── Production Forecast Metrics (non-overlapping, stride={horizon}) ──")
    print(f"Windows : {len(preds)} day-ahead forecasts")
    print_metrics(mae, rmse, mape, mae_per_horizon)

    return {
        'preds'           : preds_orig,        # (N_days, 24) — consistent with test_model
        'actuals'         : actuals_orig,      # (N_days, 24)
        'mae'             : mae,
        'rmse'            : rmse,
        'mape'            : mape,
        'mae_per_horizon' : mae_per_horizon,
        'n_windows'       : len(preds),
    }

In [ ]:
print("\n── Non-Overlapping Version (Production Simulation) ──")
prod = forecast_production(trained_model, X_test, y_test, tgt_scaler, device=device)

## 13. Visualization 
Four plots: the first 30 days of the test period (unrolled predictions), a sample 24-hour forecast horizon, the residual distribution, and training/validation loss curves.

In [ ]:
def plot_results(results, history, plot_days=30):
    preds   = results['preds']
    actuals = results['actuals']

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle('Day-Ahead Price Forecast -- Transformer', fontsize=14, fontweight='bold')

    # First N days of test period (sequences unrolled)
    ax        = axes[0, 0]
    n_hours   = plot_days * 24
    n         = min(n_hours, len(preds.ravel()))
    actual_days = n // 24  # how many complete days are actually shown
    ax.plot(actuals.ravel()[:n], label='Actual',    alpha=0.8, linewidth=0.8)
    ax.plot(preds.ravel()[:n],   label='Predicted', alpha=0.8, linewidth=0.8)
    ax.set_title(f'First {actual_days} days of test period')
    ax.set_xlabel('Hour')
    ax.set_ylabel('Price (EUR/MWh)')
    ax.legend()

    # Sample 24h forecast horizon
    ax  = axes[0, 1]
    idx = 200
    ax.plot(range(HORIZON), actuals[idx], marker='o', label='Actual',    linewidth=1.5)
    ax.plot(range(HORIZON), preds[idx],   marker='x', label='Predicted', linewidth=1.5)
    ax.set_title(f'Sample 24 h horizon (sequence #{idx})')
    ax.set_xlabel('Hour ahead')
    ax.set_ylabel('Price (EUR/MWh)')
    ax.legend()

    # Residual distribution
    ax     = axes[1, 0]
    errors = (preds - actuals).ravel()
    ax.hist(errors, bins=60, edgecolor='black', linewidth=0.4)
    ax.axvline(0, color='red', linestyle='--', linewidth=1.2,
            label=f'Zero error (perfect)')
    ax.axvline(results['mae'], color='orange', linestyle='--', linewidth=1.2,
            label=f'MAE = ±{results["mae"]:.2f} EUR/MWh')
    ax.axvline(-results['mae'], color='orange', linestyle='--', linewidth=1.2)
    ax.set_title('Residual distribution')
    ax.set_xlabel('Error (EUR/MWh)')
    ax.set_ylabel('Count')
    ax.legend()

    # ── Bottom right: Training history ───────────────────────────────────────
    ax = axes[1, 1]
    ax.plot(history['train_loss'], label=f'Train ({LOSS_FN.upper()})')
    ax.plot(history['val_loss'],   label=f'Val ({LOSS_FN.upper()})')
    ax.set_title(f'Training history  |  loss={LOSS_FN.upper()}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(f'{LOSS_FN.upper()} loss (scaled)')
    ax.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
plot_results(prod, history, plot_days=30)

### Results by season 

In [ ]:
def evaluate_by_season(results, test_df, stride=1):
    preds   = results['preds']
    actuals = results['actuals']
    n       = len(preds)

    times = pd.to_datetime(test_df['time'].values)

    # generate start indices based on stride
    starts = list(range(WINDOW_SIZE,
                        WINDOW_SIZE + n * stride,
                        stride))[:n]
    times  = times[starts]

    month  = pd.Series(times).dt.month

    seasons = {
        'Winter' : month.isin([12, 1, 2]),
        'Spring' : month.isin([3, 4, 5]),
        'Summer' : month.isin([6, 7, 8]),
        'Autumn' : month.isin([9, 10, 11]),
    }

    print(f"\n── Seasonal Error Breakdown (stride={stride}) ───────────────")
    print(f"{'Season':<10} {'N':>6} {'MAE':>8} {'RMSE':>8} {'MAPE':>8}")
    print(f"{'─'*44}")

    season_results = {}
    for season, mask in seasons.items():
        s_preds   = preds[mask.values].ravel()
        s_actuals = actuals[mask.values].ravel()

        mae  = float(np.mean(np.abs(s_preds - s_actuals)))
        rmse = float(np.sqrt(np.mean((s_preds - s_actuals) ** 2)))
        mape = float(np.mean(np.abs((s_actuals - s_preds) /
                                     np.abs(s_actuals).clip(min=1.0))) * 100)

        print(f"{season:<10} {mask.sum():>6,} {mae:>8.3f} {rmse:>8.3f} {mape:>7.2f}%")

        season_results[season] = {
            'mae' : mae,
            'rmse': rmse,
            'mape': mape,
            'n'   : int(mask.sum())
        }

    print(f"{'─'*44}")
    print(f"{'Overall':<10} {n:>6,} "
          f"{results['mae']:>8.3f} "
          f"{results['rmse']:>8.3f} "
          f"{results['mape']:>7.2f}%")

    return season_results



def plot_seasonal(season_results):
    seasons = list(season_results.keys())
    maes    = [season_results[s]['mae']  for s in seasons]
    rmses   = [season_results[s]['rmse'] for s in seasons]
    mapes   = [season_results[s]['mape'] for s in seasons]

    x = np.arange(len(seasons))
    width = 0.25

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle('Forecast Error by Season — 2018 Test Set',
                 fontsize=13, fontweight='bold')

    for ax, values, label, color in zip(
        axes,
        [maes, rmses, mapes],
        ['MAE (EUR/MWh)', 'RMSE (EUR/MWh)', 'MAPE (%)'],
        ['steelblue', 'darkorange', 'seagreen']
    ):
        bars = ax.bar(x, values, color=color, alpha=0.8, edgecolor='black', linewidth=0.5)
        ax.set_xticks(x)
        ax.set_xticklabels(seasons)
        ax.set_ylabel(label)
        ax.set_title(label)
        ax.grid(axis='y', alpha=0.3)

        # value labels on bars
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.05,
                    f'{val:.2f}', ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.show()

In [ ]:
# overlapping
season_results = evaluate_by_season(prod, test_df, stride=24)

In [ ]:
plot_seasonal(season_results)

### Price (day-ahead; actual) / Load by season [Descriptive Plots] 

### Main Outputs 

In [ ]:
mins, secs = divmod(int(elapsed), 60)
if device.type == 'cuda':
    device_label = f"cuda ({torch.cuda.get_device_name(0)})"
elif device.type == 'mps':
    device_label = "mps (Apple Silicon)"
else:
    device_label = "cpu"

print(f"\nTotal training time: {mins}m {secs}s  |  device: {device_label}")

In [ ]:
results = prod.copy()

print(f"MAE  : {results['mae']:.3f} EUR/MWh  (on average, predictions are off by {results['mae']:.1f} EUR/MWh)")
print(f"RMSE : {results['rmse']:.3f} EUR/MWh  (higher than MAE means occasional big misses)")
print(f"MAPE : {results['mape']:.2f}%          (on average, predictions are off by {results['mape']:.1f}% of the actual price)")